<a href="https://colab.research.google.com/github/sanyasana1/MN/blob/main/%D0%9B%D0%A0_9_%D0%9B%D0%B0%D0%B7%D0%B0%D1%80%D0%B5%D0%BD%D0%BA%D0%BE_3_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Лазаренко Олександр Віталійович  
**Група:** ФІТ 3-9  
**Дата:** 11.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
rating_stats = ratings_df['rating'].agg(['min', 'max', 'mean'])

print('Кількість унікальних користувачів:', n_users)
print('Кількість унікальних фільмів:', n_items)
print('Мінімальна оцінка:', rating_stats['min'])
print('Максимальна оцінка:', rating_stats['max'])
print('Середня оцінка:', round(rating_stats['mean'], 3))
print('Пропущені значення:')
display(ratings_df.isna().sum())
print('Розподіл оцінок:')
display(ratings_df['rating'].value_counts().sort_index().rename_axis('Оцінка').to_frame('Кількість'))

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53
Пропущені значення:


,0
user_id,0
item_id,0
rating,0
timestamp,0


Розподіл оцінок:


,Кількість
Оцінка,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Короткі спостереження щодо даних:**

Набір MovieLens 100K містить 100 000 оцінок, виставлених 943 користувачами для 1 682 фільмів. Оцінки мають діапазон від 1 до 5; середнє значення становить приблизно 3,53, тобто середня оцінка є вищою за середину шкали.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)
testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)
print('Кількість записів у train_data:', len(train_df))
print('Кількість записів у testset:', len(testset))

Кількість записів у train_data: 80000
Кількість записів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD параметр `n_factors` визначає кількість прихованих факторів, `lr_all` задає швидкість навчання, а `reg_all` контролює регуляризацію та допомагає зменшити перенавчання. Перевіряються 50 і 100 факторів, дві швидкості навчання та два рівні регуляризації.

Для KNNBasic параметр `k` задає максимальну кількість сусідів, `min_k` — мінімальну кількість сусідів для прогнозу. Метрики `cosine` і `pearson` дають змогу порівняти способи визначення схожості, а `user_based` перемикає пошук схожості між користувачами або між фільмами. Остаточний вибір параметрів здійснюється за найменшим CV RMSE.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)

In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)

In [ ]:
print('SVD:')
print('  Найкращий CV RMSE:', round(grid_search_svd.best_score['rmse'], 4))
print('  Найкращий CV MAE: ', round(grid_search_svd.best_score['mae'], 4))
print('  Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nKNNBasic:')
print('  Найкращий CV RMSE:', round(grid_search_knn.best_score['rmse'], 4))
print('  Найкращий CV MAE: ', round(grid_search_knn.best_score['mae'], 4))
print('  Параметри за RMSE:', grid_search_knn.best_params['rmse'])

SVD:
  Найкращий CV RMSE: 0.9428
  Найкращий CV MAE:  0.7436
  Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic:
  Найкращий CV RMSE: 1.0275
  Найкращий CV MAE:  0.8149
  Параметри за RMSE: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

Порівняйте значення, надруковані в комірці 5.3. Менше значення RMSE означає меншу середньоквадратичну помилку прогнозу, а менше MAE — меншу середню абсолютну помилку. Для формування рекомендацій використовується модель із найменшим CV RMSE; її параметри наведені у виведенні `best_params['rmse']`.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_knn_params = grid_search_knn.best_params['rmse'].copy()
trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params, verbose=False)
best_svd.fit(trainset)
best_knn.fit(trainset)

## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD      — RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}')
print(f'KNNBasic — RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}')

SVD      — RMSE: 0.9192, MAE: 0.7234
KNNBasic — RMSE: 1.0117, MAE: 0.7993


In [ ]:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
])
results = results.sort_values(by='RMSE').reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011708,0.799260


**Таблиця тестових результатів:**

Фактичні значення RMSE та MAE для обох моделей виводяться у таблиці `results` у комірці 7.3.

**Порівняння з CV:** зіставте тестові RMSE/MAE з відповідними CV-значеннями, надрукованими у комірці 5.3. Невелика різниця є очікуваною через випадковий поділ даних; помітно гірші тестові показники можуть свідчити про нестабільність моделі або відмінності між навчальною і тестовою частинами.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
user_id = '196'

if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    recommendation_model = best_svd
    recommendation_model_name = 'SVD'
else:
    recommendation_model = best_knn
    recommendation_model_name = 'KNNBasic'
print('Модель для рекомендацій:', recommendation_model_name)

try:
    inner_uid = trainset.to_inner_uid(str(user_id))
except ValueError:
    inner_uid = None
    print(f'Користувача з ID {user_id} немає у навчальній вибірці.')

if inner_uid is not None:
    rated_inner_iids = {
        inner_iid for inner_iid, rating in trainset.ur[inner_uid]
    }

    candidate_inner_iids = [
        inner_iid for inner_iid in range(trainset.n_items)
        if inner_iid not in rated_inner_iids
    ]

    recommendations = []
    for inner_iid in candidate_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        prediction = recommendation_model.predict(
            uid=str(user_id), iid=raw_iid
        )
        recommendations.append({
            'item_id': raw_iid,
            'predicted_rating': prediction.est
        })

    top_10_recommendations = (
        pd.DataFrame(recommendations)
        .sort_values('predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    print(f'Топ-10 рекомендацій для користувача {user_id}:')
    display(top_10_recommendations)

Модель для рекомендацій: SVD
Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

In [ ]:
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    cv_winner = 'SVD'
    cv_rmse = grid_search_svd.best_score['rmse']
    cv_mae = grid_search_svd.best_score['mae']
    winner_params = grid_search_svd.best_params['rmse']
    winner_test_rmse, winner_test_mae = svd_rmse, svd_mae
else:
    cv_winner = 'KNNBasic'
    cv_rmse = grid_search_knn.best_score['rmse']
    cv_mae = grid_search_knn.best_score['mae']
    winner_params = grid_search_knn.best_params['rmse']
    winner_test_rmse, winner_test_mae = knn_rmse, knn_mae

print(
    f"У лабораторній роботі було побудовано та порівняно моделі SVD і KNNBasic "
    f"на наборі MovieLens 100K, який містить {len(ratings_df)} оцінок, "
    f"{ratings_df['user_id'].nunique()} користувачів і "
    f"{ratings_df['item_id'].nunique()} фільмів."
)
print(
    f"За результатами трьохфолдної кросвалідації кращою за RMSE стала модель "
    f"{cv_winner} з CV RMSE = {cv_rmse:.4f} та CV MAE = {cv_mae:.4f}."
)
print(f"Найкращі параметри цієї моделі: {winner_params}.")
print(
    f"На незалежній тестовій вибірці обрана модель отримала "
    f"RMSE = {winner_test_rmse:.4f} та MAE = {winner_test_mae:.4f}."
)
print(
    f"Різниця між тестовим і CV RMSE становить "
    f"{abs(winner_test_rmse - cv_rmse):.4f}; її слід оцінювати з урахуванням "
    f"того, що CV та тестування виконуються на різних частинах даних."
)
print(
    "Для користувача сформовано топ-10 фільмів, які не входили до його оцінок "
    "у навчальній вибірці. Рекомендації залежать від наявної історії оцінок, "
    "тому для нових користувачів або фільмів виникає проблема холодного старту."
)

У лабораторній роботі було побудовано та порівняно моделі SVD і KNNBasic на наборі MovieLens 100K, який містить 100000 оцінок, 943 користувачів і 1682 фільмів.
За результатами трьохфолдної кросвалідації кращою за RMSE стала модель SVD з CV RMSE = 0.9428 та CV MAE = 0.7436.
Найкращі параметри цієї моделі: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}.
На незалежній тестовій вибірці обрана модель отримала RMSE = 0.9192 та MAE = 0.7234.
Різниця між тестовим і CV RMSE становить 0.0236; її слід оцінювати з урахуванням того, що CV та тестування виконуються на різних частинах даних.
Для користувача сформовано топ-10 фільмів, які не входили до його оцінок у навчальній вибірці. Рекомендації залежать від наявної історії оцінок, тому для нових користувачів або фільмів виникає проблема холодного старту.


**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**